# Full Ingestion — STATS19 Road Safety, All 5 Years, All 3 Tables → SQLite

`data_workflow.ipynb` established the ingestion method on a single table
(collision) narrowed to a handful of analysis columns: load the full-width
table, use the DfT data guide to programmatically detect which columns are
genuinely coded categorical fields, clean the `-1` missing/unknown sentinel,
and decode codes to human-readable labels via a guide-driven lookup merge.

This notebook applies that same method at full scope: **all three STATS19
tables** (collision, vehicle, casualty), **full width** (every column, not
just an analysis subset), for **all 5 years** already present in the
`*-last-5-years.csv` extracts, loaded into a **single SQLite database**.

It's a two-step process per table, mirroring the coded → decoded split
`data_workflow.ipynb` demonstrated:

1. **Normalised** — the full-width table as delivered, with the `-1`
   sentinel cleaned to a true `NULL` but the DfT integer codes left in
   place (matching the coded `id` columns the Power BI model's `Dim_*`
   tables key against — see `relationships.tmdl`).
2. **Denormalised** — every coded column decoded to its text label via the
   data guide, so the table is flat and self-describing: no joins to a
   dimension table are needed to read it.

Both stages are written to SQLite so the coded/FK-style tables stay
available (useful if the star schema from the Power BI project is ever
wanted directly), but the **denormalised tables are the primary, indexed,
query-ready surface** — that's the end goal.

### Prerequisite: raw data

This notebook expects three raw CSVs already present in `data/raw/` — they
are **not included in this repo** (too large). Download the "last 5 years"
collision, vehicle, and casualty extracts from the DfT's STATS19 Road Safety
Open Data page: <https://www.gov.uk/government/statistical-data-sets/road-safety-open-data>,
and place them at:

```
data/raw/dft-road-casualty-statistics-collision-last-5-years.csv
data/raw/dft-road-casualty-statistics-vehicle-last-5-years.csv
data/raw/dft-road-casualty-statistics-casualty-last-5-years.csv
```

The data guide (`data/dimensions/dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx`)
*is* included in the repo — it's small enough to commit and this notebook
depends on it directly for decoding.

In [1]:
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd

## Paths & constants

Three raw extracts, one data guide, one output database. `FIELD_NAME_OVERRIDES`
records the one field-name mismatch between the data guide and the raw CSV
headers that `data_workflow.ipynb` already found by inspecting the full
collision table (`enhanced_severity_collision` in the CSV vs.
`enhanced_collision_severity` in the guide — the words are swapped). The same
check against the vehicle and casualty tables below turns up no further
mismatches worth a manual override (the guide vs. raw-header diff for those
two tables is limited to the `_adjusted_serious`/`_adjusted_slight` weighting
fields, which aren't coded categoricals — they're single-row, non-integer
guide entries and are excluded automatically by `detect_coded_fields`).

`ID_STRING_COLUMNS` covers identifier/code columns that must **not** be left
to pandas' numeric type inference: `collision_ref_no` in particular contains
values like `"070326701"` where a leading zero is significant — inferred as
`int64` it would silently become `70326701`.

In [2]:
DATA_DIR = Path("data")
DATA_GUIDE_PATH = DATA_DIR / "dimensions" / "dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx"
DB_PATH = DATA_DIR / "road_safety.db"

TABLES = ["collision", "vehicle", "casualty"]
RAW_PATH_TEMPLATE = "data/raw/dft-road-casualty-statistics-{table}-last-5-years.csv"

FIELD_NAME_OVERRIDES = {
    "collision": {"enhanced_severity_collision": "enhanced_collision_severity"},
}

# `detect_coded_fields` requires every code to be int-like, which correctly
# excludes continuous fields that merely carry a documented `-1` sentinel
# (e.g. `first_road_number`) but also wrongly excludes fields that ARE
# genuinely coded, just not with integers. `local_authority_ons_district`
# and `local_authority_highway` use alphanumeric ONS GSS codes (e.g.
# "E06000002") and are missed by the heuristic even though the guide has
# full label lookups for them (423 and 220 rows respectively) — see the
# note below Step 1.
ADDITIONAL_CODED_FIELDS = {
    "collision": {
        "local_authority_ons_district": "local_authority_ons_district",
        "local_authority_highway": "local_authority_highway",
    },
}

ID_STRING_COLUMNS = [
    "collision_index", "collision_ref_no",
    "local_authority_ons_district", "local_authority_highway", "local_authority_highway_current",
    "lsoa_of_accident_location", "lsoa_of_driver", "lsoa_of_casualty",
]

In [3]:
data_guide = pd.read_excel(DATA_GUIDE_PATH, sheet_name="2024_code_list")
data_guide["table"].value_counts()

table
collision               1418
vehicle                  265
casualty                 128
historical_revisions      10
Name: count, dtype: int64

## Detecting coded fields (reused from `data_workflow.ipynb`, generalised)

`is_int_like` and the coded-field detection logic are unchanged from
`data_workflow.ipynb`. The one generalisation: `data_workflow.ipynb` only
needed the *set* of coded field names for one table. Here, decoding runs
against every coded column across three tables, so `detect_coded_fields`
returns a `{raw_column_name: guide_field_name}` mapping instead — identity
for every field except the one overridden name above — which both drives
the decode step and records, for each table, exactly which of its raw
columns are genuinely coded categoricals vs. continuous/free-format/ID
fields that merely carry a documented `-1` sentinel.

In [4]:
def is_int_like(value):
    """Return True if `value` parses as an integer (a discrete code)."""
    try:
        int(value)
        return True
    except (ValueError, TypeError):
        return False


def detect_coded_fields(guide, raw_columns, overrides=None):
    """Map raw columns to their data-guide field name, for genuinely coded fields.

    A guide field qualifies as a coded categorical if it has more than one
    row AND every `code/format` entry is an integer (see `data_workflow.ipynb`
    for why: this excludes continuous/free-format fields that merely have a
    documented sentinel noted alongside a range, e.g. "1 to 9999" plus a
    "-1 = Unknown" note).

    Parameters
    ----------
    guide : pd.DataFrame
        The data guide, already filtered to one table.
    raw_columns : list of str
        Column names as they actually appear in the raw CSV.
    overrides : dict, optional
        `{raw_column_name: guide_field_name}` for the handful of fields
        where the CSV header doesn't match the guide's field name.

    Returns
    -------
    dict
        `{raw_column_name: guide_field_name}` for every raw column that is
        a genuine coded categorical.
    """
    overrides = overrides or {}
    coded_guide_fields = {
        field
        for field, group in guide.groupby("field name")
        if len(group) > 1 and group["code/format"].apply(is_int_like).all()
    }
    return {
        col: overrides.get(col, col)
        for col in raw_columns
        if overrides.get(col, col) in coded_guide_fields
    }

## Cleaning the `-1` sentinel (reused verbatim from `data_workflow.ipynb`)

Applied to the full-width table, this now cleans every integer column with
a documented `-1` sentinel — both the coded categoricals (whose `-1` will
decode to "Data missing or out of range" in the denormalised step, since
the guide documents that as an explicit label for most fields) and plain
numeric fields like `engine_capacity_cc` or `age_of_driver` that only carry
`-1` as an "unknown" flag with no dimension table behind it. This is the
**normalised** table.

In [5]:
def handle_missing_sentinels(df, sentinel=-1):
    """Convert STATS19's missing/unknown sentinel code to a proper NaN.

    STATS19 encodes "missing or unknown" on integer-coded fields as a literal
    ``-1`` value rather than a null. Left as-is, ``-1`` counts as "valid" data
    under pandas' native missing-value tooling. This function scans every
    integer column for the sentinel and, where found, casts that column to a
    nullable integer dtype and replaces the sentinel with ``pd.NA``.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with STATS19-coded integer columns.
    sentinel : int, default -1
        The STATS19 missing/unknown sentinel value.

    Returns
    -------
    pd.DataFrame
        Copy of ``df`` with the sentinel replaced by NaN on affected columns.
    """
    df = df.copy()
    int_cols = df.select_dtypes(include=["int8", "int16", "int32", "int64"]).columns
    for col in int_cols:
        if (df[col] == sentinel).any():
            df[col] = df[col].astype("Int64").replace(sentinel, pd.NA)
    return df

## Decoding coded fields (reused from `data_workflow.ipynb`, generalised)

Same merge-based decode as `data_workflow.ipynb`'s `decode_categorical_codes`,
generalised from a hardcoded two-field tuple to the full `coded_map` detected
above, so every coded column in a table gets decoded, not just a couple
picked for one analysis. This produces the **denormalised** table.

One addition `data_workflow.ipynb` didn't need: the lookup built from the
data guide is de-duplicated before merging. Decoding every field surfaced a
handful of exact duplicate rows in the guide itself (two fields, one
duplicate code each) that fan out into duplicate rows on merge — narrow
enough to have never mattered for the two fields `data_workflow.ipynb`
decoded, but real once every coded column is in play.

In [6]:
def decode_categorical_codes(df, guide, coded_map):
    """Decode every coded categorical field in `coded_map` via the data guide.

    For each `raw_column: guide_field` pair, builds a lookup straight from
    the official data guide and merges it onto `df`, replacing the integer
    code with its human-readable label. Rows already converted to `pd.NA`
    by `handle_missing_sentinels` don't match any lookup code and correctly
    decode to a true missing value rather than a stray "-1" label.

    The data guide itself contains a handful of exact duplicate rows (e.g.
    `local_authority_ons_district` code `E09000001` "City of London" and
    `local_authority_district` code `143` "Castle Morpeth" both appear
    twice) — left as-is, a duplicate lookup row silently fans out into a
    duplicate row per match in the merge, so the lookup is de-duplicated
    before merging.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe containing the coded columns named as keys in `coded_map`.
    guide : pd.DataFrame
        The data guide, already filtered to this table.
    coded_map : dict
        `{raw_column_name: guide_field_name}` from `detect_coded_fields`.

    Returns
    -------
    pd.DataFrame
        Copy of `df` with each coded column replaced by its label (category dtype).
    """
    df = df.copy()
    for raw_col, guide_field in coded_map.items():
        lookup = guide.loc[guide["field name"] == guide_field, ["code/format", "label"]]
        lookup = lookup.rename(columns={"code/format": raw_col, "label": f"{raw_col}_label"})
        lookup[raw_col] = lookup[raw_col].astype(str)
        lookup = lookup.drop_duplicates(subset=[raw_col])

        df[raw_col] = df[raw_col].astype(str)
        df = df.merge(lookup, on=raw_col, how="left")
        df[raw_col] = df.pop(f"{raw_col}_label").astype("category")
    return df

## Temporal features (collision only, adapted from `data_workflow.ipynb`)

Only the collision table carries `date`/`time` columns — vehicle and
casualty rows join back to it on `collision_index` for temporal context.
`date` is converted to an ISO `YYYY-MM-DD` string (so SQLite's built-in
date functions work directly on it) and `hour`/`month` are derived, exactly
as `data_workflow.ipynb`'s `derive_temporal_features` did. `day_of_week` is
left untouched here — it's already decoded via the guide lookup above, so
re-deriving it would be redundant.

In [7]:
def derive_temporal_features(df):
    """Normalise `date` to ISO format and derive `hour`/`month` from date/time.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe containing `time` (`HH:MM` strings) and `date`
        (`DD/MM/YYYY` strings) columns.

    Returns
    -------
    pd.DataFrame
        Copy of `df` with `date` rewritten to `YYYY-MM-DD` and new `hour`
        and `month` integer columns added.
    """
    df = df.copy()
    parsed_date = pd.to_datetime(df["date"], format="%d/%m/%Y")
    df["hour"] = pd.to_datetime(df["time"], format="%H:%M").dt.hour
    df["month"] = parsed_date.dt.month
    df["date"] = parsed_date.dt.strftime("%Y-%m-%d")
    return df

## Step 1 — load and clean each table (normalised)

Full width, every column, for all three tables. `ID_STRING_COLUMNS` is
passed as an explicit `dtype` override so identifier columns keep leading
zeros and mixed alphanumeric values intact; everything else is left to
pandas' own inference, then `handle_missing_sentinels` cleans every integer
column that carries the `-1` sentinel.

**A second data-guide gotcha, found by checking the denormalised output:**
`local_authority_ons_district` and `local_authority_highway` were coming
out of the first version of this pipeline still as raw ONS codes (e.g.
`"E06000002"`) instead of names, in every row. The cause: `detect_coded_fields`'s
"every code is an integer" rule — right for filtering out continuous fields
like `first_road_number` — also silently drops these two fields, because
their codes are alphanumeric, not integers. The data guide does document
full lookups for both (423 rows for `local_authority_ons_district`, 220 for
`local_authority_highway`), so `ADDITIONAL_CODED_FIELDS` above adds them back
in explicitly rather than relaxing the general heuristic (which is still
correct for everything else — checked across all three tables, these are
the only two genuinely-coded fields it misses).

Separately, `local_authority_district` *is* decoding correctly where it has
a value, but only 194 of the collision table's 513,801 rows have one — the
field was effectively retired in favour of the ONS-coded version, so the
rest are a genuine data gap rather than a decode failure.

In [8]:
guides = {name: data_guide[data_guide["table"] == name].copy() for name in TABLES}
coded_maps = {}
normalised = {}

for name in TABLES:
    t0 = time.time()
    raw_path = RAW_PATH_TEMPLATE.format(table=name)
    raw_columns = pd.read_csv(raw_path, nrows=0).columns.tolist()

    coded_maps[name] = detect_coded_fields(guides[name], raw_columns, FIELD_NAME_OVERRIDES.get(name))
    coded_maps[name].update(ADDITIONAL_CODED_FIELDS.get(name, {}))

    dtype_overrides = {col: "str" for col in ID_STRING_COLUMNS if col in raw_columns}
    df = pd.read_csv(raw_path, dtype=dtype_overrides, low_memory=False)
    normalised[name] = handle_missing_sentinels(df)

    print(f"{name:10s} {normalised[name].shape[0]:>7,} rows x {normalised[name].shape[1]:>2} cols "
          f"({len(coded_maps[name])} coded fields) — {time.time() - t0:.1f}s")

collision  513,801 rows x 44 cols (26 coded fields) — 1.4s


vehicle    937,265 rows x 32 cols (23 coded fields) — 1.7s


casualty   652,821 rows x 23 cols (14 coded fields) — 0.9s


In [9]:
normalised["collision"].info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 513801 entries, 0 to 513800
Data columns (total 44 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   collision_index                                   513801 non-null  object 
 1   collision_year                                    513801 non-null  int64  
 2   collision_ref_no                                  513801 non-null  object 
 3   location_easting_osgr                             513748 non-null  float64
 4   location_northing_osgr                            513748 non-null  float64
 5   longitude                                         513748 non-null  float64
 6   latitude                                          513748 non-null  float64
 7   police_force                                      513801 non-null  int64  
 8   collision_severity                                513801 non-null  int64  
 9   numb

## Step 2 — decode to build the denormalised tables

Each normalised table is decoded against its own table-scoped data guide.
The collision table additionally gets its date/time features normalised.

In [10]:
denormalised = {}

for name in TABLES:
    t0 = time.time()
    df = decode_categorical_codes(normalised[name], guides[name], coded_maps[name])
    if name == "collision":
        df = derive_temporal_features(df)
    denormalised[name] = df
    print(f"{name:10s} decoded — {time.time() - t0:.1f}s")

denormalised["collision"][["collision_severity", "day_of_week", "weather_conditions", "date", "hour", "month"]].head()

collision  decoded — 5.1s


vehicle    decoded — 6.2s


casualty   decoded — 2.2s


,collision_severity,day_of_week,weather_conditions,date,hour,month
0,Slight,Saturday,Fine no high winds,2025-02-15,19,2
1,Serious,Tuesday,Fine no high winds,2024-10-22,14,10
2,Slight,Friday,Raining no high winds,2025-12-19,6,12
3,Fatal,Tuesday,Fine no high winds,2025-04-22,20,4
4,Slight,Monday,Fine no high winds,2025-04-28,17,4


In [11]:
denormalised["vehicle"][["vehicle_type", "sex_of_driver", "journey_purpose_of_driver"]].head()

,vehicle_type,sex_of_driver,journey_purpose_of_driver
0,Car,Male,Commuting to or from work
1,Car,Female,Journey as part of work
2,Car,Male,Commuting to or from work
3,Car,Male,Not known or not requested
4,Car,Female,Education and educational escort


In [12]:
denormalised["casualty"][["casualty_class", "casualty_severity", "casualty_type"]].head()

,casualty_class,casualty_severity,casualty_type
0,Pedestrian,Slight,Pedestrian
1,Pedestrian,Slight,Pedestrian
2,Pedestrian,Slight,Pedestrian
3,Pedestrian,Slight,Pedestrian
4,Pedestrian,Slight,Pedestrian


## Step 3 — write to SQLite and build indexes

Both the normalised and denormalised tables are written, so the coded
star-schema-style tables stay available alongside the flat, query-ready
denormalised ones (`collision`, `vehicle`, `casualty` — no `_normalised`
suffix).

Indexes mirror the natural keys implied by `relationships.tmdl`: `collision`
is keyed on `collision_index`; `vehicle` and `casualty` key on
`collision_index` plus their own reference columns (matching how
`relationships.tmdl` joins `'Vehicle Provisional Mid-2024'.collision_index`
and `'Casualty Provisional Mid-2024'.collision_index` back to the collision
table), with a secondary index on `collision_year` on all three since it's
present on every row and is the most common filter/group-by column.

In [13]:
INDEX_SPECS = {
    "collision": [("idx_{table}_collision_index", ["collision_index"], True),
                  ("idx_{table}_collision_year", ["collision_year"], False),
                  ("idx_{table}_date", ["date"], False)],
    "vehicle": [("idx_{table}_key", ["collision_index", "vehicle_reference"], True),
                ("idx_{table}_collision_index", ["collision_index"], False),
                ("idx_{table}_collision_year", ["collision_year"], False)],
    "casualty": [("idx_{table}_key", ["collision_index", "vehicle_reference", "casualty_reference"], True),
                 ("idx_{table}_collision_index", ["collision_index"], False),
                 ("idx_{table}_collision_year", ["collision_year"], False)],
}


def write_and_index(conn, table_name, df, base_name):
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    for name_template, cols, unique in INDEX_SPECS[base_name]:
        idx_name = name_template.format(table=table_name)
        unique_kw = "UNIQUE " if unique else ""
        cols_sql = ", ".join(cols)
        conn.execute(f"CREATE {unique_kw}INDEX {idx_name} ON {table_name}({cols_sql})")


DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)

for name in TABLES:
    write_and_index(conn, f"{name}_normalised", normalised[name], name)
    write_and_index(conn, name, denormalised[name], name)

conn.execute("ANALYZE")
conn.commit()
print(f"Written to {DB_PATH.resolve()}")

Written to /home/simon/AI Masters/Capstone Projects/agentic-ai/data/road_safety.db


## Verification

Row counts in the database should match the source CSVs exactly (no rows
dropped or duplicated across the normalised/denormalised split), every
planned index should exist, and a query joining all three denormalised
tables should run without any manual decoding — proving the database is
ready for querying as-is.

In [14]:
print(f"{'table':22s} {'db rows':>10s} {'csv rows':>10s}")
for name in TABLES:
    csv_rows = sum(1 for _ in open(RAW_PATH_TEMPLATE.format(table=name))) - 1
    for table_name in (f"{name}_normalised", name):
        db_rows = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        match = "OK" if db_rows == csv_rows else "MISMATCH"
        print(f"{table_name:22s} {db_rows:>10,} {csv_rows:>10,}  {match}")

table                     db rows   csv rows
collision_normalised      513,801    513,801  OK
collision                 513,801    513,801  OK
vehicle_normalised        937,265    937,265  OK
vehicle                   937,265    937,265  OK
casualty_normalised       652,821    652,821  OK
casualty                  652,821    652,821  OK


In [15]:
tables_in_db = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
indexes_in_db = pd.read_sql("SELECT name, tbl_name FROM sqlite_master WHERE type='index' ORDER BY tbl_name, name", conn)
display(tables_in_db)
display(indexes_in_db)

,name
0,casualty
1,casualty_normalised
2,collision
3,collision_normalised
4,sqlite_stat1
5,vehicle
6,vehicle_normalised


,name,tbl_name
0,idx_casualty_collision_index,casualty
1,idx_casualty_collision_year,casualty
2,idx_casualty_key,casualty
3,idx_casualty_normalised_collision_index,casualty_normalised
4,idx_casualty_normalised_collision_year,casualty_normalised
5,idx_casualty_normalised_key,casualty_normalised
6,idx_collision_collision_index,collision
7,idx_collision_collision_year,collision
8,idx_collision_date,collision
9,idx_collision_normalised_collision_index,collision_normalised


In [16]:
query = """
SELECT
    c.collision_year,
    c.collision_severity,
    v.vehicle_type,
    COUNT(DISTINCT c.collision_index) AS collisions,
    COUNT(DISTINCT ca.rowid) AS casualties
FROM collision AS c
JOIN vehicle AS v ON v.collision_index = c.collision_index
LEFT JOIN casualty AS ca ON ca.collision_index = c.collision_index
WHERE c.collision_severity = 'Fatal'
GROUP BY c.collision_year, c.collision_severity, v.vehicle_type
ORDER BY c.collision_year DESC, collisions DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,collision_year,collision_severity,vehicle_type,collisions,casualties
0,2025,Fatal,Car,1065,1887
1,2025,Fatal,Motorcycle over 500cc,271,347
2,2025,Fatal,Van / Goods 3.5 tonnes mgw or under,188,300
3,2025,Fatal,Goods 7.5 tonnes mgw and over,154,245
4,2025,Fatal,Pedal cycle,86,101
5,2025,Fatal,Motorcycle 125cc and under,70,88
6,2025,Fatal,Bus or coach (17 or more pass seats),42,108
7,2025,Fatal,Motorcycle over 125cc and up to 500cc,36,47
8,2025,Fatal,Other vehicle,28,41
9,2025,Fatal,Agricultural vehicle,24,35


In [17]:
conn.close()

## Summary

`data/road_safety.db` now holds all five years of STATS19 collision,
vehicle, and casualty data, twice over: a normalised (coded) copy for each
table matching the DfT/Power BI source encoding, and a denormalised
(decoded, indexed) copy — `collision`, `vehicle`, `casualty` — that's flat
and self-describing enough to query directly with plain SQL, no dimension
joins required.